In [ ]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "anomalib==2.7.0",
        "timm==1.0.30",
        "lightning==2.6.6",
    ],
    check=True,
)

print("Dependencies installed: OK")

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/MRdrissi/industrial-defect-ai.git"
REPO_DIR = Path("/kaggle/working/industrial-defect-ai")
SOURCE_TAG = "exp/patchcore-bottle-exp003"

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(REPO_DIR), "fetch", "origin", "--tags"],
    check=True,
)

subprocess.run(
    [
        "git", "-C", str(REPO_DIR),
        "checkout", "--detach", f"refs/tags/{SOURCE_TAG}",
    ],
    check=True,
)

source_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

sys.path.insert(0, str(REPO_DIR))

from src.inference.patchcore_inference import PatchCoreInference

print("Source commit:", source_commit)
print("Source tag:", SOURCE_TAG)
print("PatchCoreInference import: OK")

In [ ]:
import hashlib

INPUT_ROOT = Path("/kaggle/input")

EXPECTED_SHA256 = (
    "dcf0f8df3dc9b4be9ee1bcb897d3ec6f24a3191240dcfd4e68afd598acd77cde"
)

checkpoints = [
    path for path in INPUT_ROOT.rglob("patchcore_model.pt")
    if path.parent.name == "EXP-002"
       and path.parent.parent.name == "bottle"
]

assert len(checkpoints) == 1, (
    "Attacher les Outputs du notebook GPU 14 (EXP-002)."
)

checkpoint_path = checkpoints[0]

actual_sha256 = hashlib.sha256(
    checkpoint_path.read_bytes()
).hexdigest()

assert actual_sha256 == EXPECTED_SHA256

print("Checkpoint:", checkpoint_path)
print("SHA-256: OK")

In [ ]:
import os
import json
import torch

from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform

config = json.loads(
    (REPO_DIR / "experiments/configs/patchcore_bottle.json")
    .read_text(encoding="utf-8")
)

dataset_root = None

for current, dirs, _ in os.walk(INPUT_ROOT):
    candidate = Path(current)
    if "bottle" in dirs and (
            candidate / "bottle/train/good"
    ).is_dir():
        dataset_root = candidate
        break

assert dataset_root is not None, "Dataset MVTec AD introuvable"

val_dataset = MVTecDataset(
    dataset_root=dataset_root,
    manifest_path=REPO_DIR / config["manifest"],
    category="bottle",
    split="val",
    transform=MVTecTransform(
        size=tuple(config["transform"]["size"]),
        profile="patchcore",
    ),
)

assert len(val_dataset) == 42

target_path = "bottle/train/good/002.png"

matching = [
    i for i, row in enumerate(val_dataset.samples)
    if row["image_path"] == target_path
]

assert len(matching) == 1
sample = val_dataset[matching[0]]

# Charger le modèle une seule fois.
engine = PatchCoreInference.from_checkpoint(
    category="bottle",
    checkpoint_path=checkpoint_path,
    device="cpu",
    expected_sha256=EXPECTED_SHA256,
)

# Inférence avec notre nouvelle API.
result = engine.predict(sample["image"])

assert tuple(engine.model.memory_bank.shape) == (1710, 1536)
assert result.anomaly_map.shape == (256, 256)
assert torch.isfinite(result.anomaly_map).all()

print("=== PATCHCORE API INFERENCE ===")
print(result.to_dict())
print("Inference: OK")

In [ ]:
import numpy as np

reference_path = (
        REPO_DIR / "docs/data/us-04-02/reference_prediction.npz"
)

with np.load(reference_path, allow_pickle=False) as reference:
    expected_score = float(reference["pred_score"].reshape(-1)[0])
    expected_map = reference["anomaly_map"].reshape(256, 256)

actual_map = result.anomaly_map.numpy()

score_diff = abs(result.score - expected_score)
map_diff = np.abs(actual_map - expected_map)
map_max_diff = float(map_diff.max())

map_range = float(expected_map.max() - expected_map.min())
assert map_range > 0
map_relative_diff = map_max_diff / map_range

print("=== COMPARISON WITH EXP-002 ===")
print("Reference score:", expected_score)
print("New API score:", result.score)
print("Score difference:", score_diff)
print("Maximum map difference:", map_max_diff)
print("Relative map difference:", map_relative_diff)

assert score_diff <= 1e-3
assert map_max_diff <= 1e-3
assert map_relative_diff <= 1e-4

print("PATCHCORE INFERENCE API: PASS")